# Droplet rebound simulation - Postprocessing

This notebook is used to postprocess the simulations for the droplet rebound test case.

In [ ]:
//#r "../../src/L4-application/BoSSSpad/bin/Release/net5.0/BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Debug/net5.0/BoSSSpad.dll"
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
// BoSSSshell.WorkflowMgm.Init("DropletRebound");
// var sessions = BoSSSshell.WorkflowMgm.Sessions;

In [ ]:
//OpenOrCreateDatabase(@"\\hpccluster\hpccluster-scratch\smuda\DropletRebound");
OpenOrCreateDatabase(@"\\dc3\userspace\smuda\hpccluster\DropletRebound");

In [ ]:
var sessions = databases.Pick(0).Sessions;
sessions

#0: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart4*	04/18/2024 11:39:26	cb667911...
#1: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart3*	04/17/2024 11:27:29	d1016ab7...
#2: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart2*	04/16/2024 11:43:08	005048e6...
#3: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_c4t4_restart3*	04/06/2024 19:28:58	a73dc751...
#4: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_c4t4_restart2*	04/05/2024 13:19:42	fd4010c4...
#5: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI10_c4t4_restart_withReInit1*	04/04/2024 11:00:35	dfb73893...
#6: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_c4t4_restart_withReInit1*	03/28/2024 13:40:49	e78efbb8...
#7: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_c4t4_restart_withReInit1*	03/26/2024 18:08:

In [ ]:
var select = sessions.Where(sess => sess.Name.Contains("DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart2"));
select

#0: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart2*	04/16/2024 11:43:08	005048e6...


In [ ]:
// var sess = select.Pick(0);
var sess = sessions.Pick(0);
sess

DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart4*	04/18/2024 11:39:26	cb667911...

In [ ]:
List<ISessionInfo> restartSessionList = new List<ISessionInfo>();
restartSessionList.Add(sess);
Guid restartID = sess.RestartedFrom;
while(restartID != Guid.Empty) {
    var restartSession = sessions.Where(sess => sess.ID == restartID).Single();
    restartSessionList.Add(restartSession);
    restartID = restartSession.RestartedFrom;
}
restartSessionList.Reverse();
restartSessionList

#0: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_withReInit10_c4t4*	03/06/2024 08:42:56	0cf1b182...
#1: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI10_c4t4_restart_withReInit1*	04/04/2024 11:00:35	dfb73893...
#2: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart2*	04/16/2024 11:43:08	005048e6...
#3: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart3*	04/17/2024 11:27:29	d1016ab7...
#4: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart4*	04/18/2024 11:39:26	cb667911...


In [ ]:
//restartSessionList.Pick(2).Timesteps

In [ ]:
List<ITimestepInfo> timestepList = new List<ITimestepInfo>();
foreach (var sess in restartSessionList) {
    int firstIndex = sess.Timesteps.Pick(0).TimeStepNumber.MajorNumber;
    int foundIndex = timestepList.FindIndex(ts => ts.TimeStepNumber.MajorNumber == firstIndex);
    if (foundIndex > -1) {
        timestepList.RemoveRange(foundIndex, timestepList.Count - foundIndex);
    }
    timestepList.AddRange(sess.Timesteps);
} 
//timestepList

In [ ]:
timestepList.Count

352

In [ ]:
timestepList.Take(10)

#0:  { Time-step: 0.0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }
#1:  { Time-step: 0.1; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, MPIrank, CutCells; Name:  }
#2:  { Time-step: 0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }
#3:  { Time-step: 1; Physical time: 1E-06s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Resi

In [ ]:
//sess.KeysAndQueries["Option_LevelSetEvolution"]
//sess.GetSessionDirectory()
//sess.RestartedFrom

\\hpccluster\hpccluster-scratch\smuda\DropletRebound\sessions\d2c5673f-901d-42d7-8d6a-856bd9e8865d

In [ ]:
//sess.Timesteps.Skip(2).Every(10).Export().WithSupersampling(2).Do()
//sess.Timesteps.Every(2).Export().WithSupersampling(2).Do()

Starting export process... Data will be written to the directory: C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\DropletRebound__DropletRebound_8x8x8AMR1_dropVelocity100vH_restart2_ReInit__b52120d3-bd87-49f5-851f-fbd1c7ebe548


C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\DropletRebound__DropletRebound_8x8x8AMR1_dropVelocity100vH_restart2_ReInit__b52120d3-bd87-49f5-851f-fbd1c7ebe548

## Droplet metrics (center of mass, volume, sphericity) over time

In [ ]:
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.XNSECommon;

In [ ]:
var sessTimesteps = timestepList.Skip(2).Every(10);
sessTimesteps.Take(5)

#0:  { Time-step: 0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }
#1:  { Time-step: 10; Physical time: 1E-05s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }
#2:  { Time-step: 20; Physical time: 2.0000000000000005E-05s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }
#3:  { Time-step: 30; Physical time: 3.000000000000001E-05

In [ ]:
int timesteps = sessTimesteps.Count();
timesteps

35

In [ ]:
double[] time = new double[timesteps];
MultidimensionalArray centerOfMass = MultidimensionalArray.Create(timesteps, 3);
double[] volume = new double[timesteps];
double[] sphericity = new double[timesteps];
double[] minDropPos = new double[timesteps];

In [ ]:
int degPhi = sessTimesteps.ElementAt(0).GetField("Phi").Basis.Degree;
int quadOrder = (2 * degPhi) * 2 + 1;   // using Saye

In [ ]:
sessTimesteps.ElementAt(0).GetField("Phi").Basis.GridDat.iGeomCells.Count

1856

In [ ]:
for (int ts = 0; ts < timesteps; ts++) {
    var tStep = sessTimesteps.ElementAt(ts);

    time[ts] = tStep.PhysicalTime;

    SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
    GridData grdData = (GridData)phi.GridDat; 
    LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
    levelSet.Acc(1.0, phi); 
    LevelSetTracker LsTrk = new LevelSetTracker(grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, levelSet);
    LsTrk.UpdateTracker(tStep.PhysicalTime);

    var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), quadOrder, 1).XQuadSchemeHelper;
    SpeciesId spcId = LsTrk.GetSpeciesId("A");
    var vqs = SchemeHelper.GetVolumeQuadScheme(spcId);


    // volume
    double dropVolume = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        vqs.Compile(LsTrk.GridDat, quadOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            EvalResult.SetAll(1.0);
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < Length; i++)
            dropVolume += ResultsOfIntegration[i, 0];
        }
    ).Execute();

    volume[ts] = dropVolume;


    // center of mass
    double[] dropCoM = new double[3];
    CellQuadrature.GetQuadrature(new int[] { 3 }, LsTrk.GridDat,
        vqs.Compile(LsTrk.GridDat, quadOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            NodeSet nodes_global = QR.Nodes.CloneAs();
            for (int i = i0; i < i0 + Length; i++) {
                LsTrk.GridDat.TransformLocal2Global(QR.Nodes, nodes_global, i);
                EvalResult.AccSubArray(1.0, nodes_global, new int[] { i - i0, -1, -1 });
            }
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for(int d = 0; d < 3; d++) {
                for(int i = 0; i < Length; i++)
                    dropCoM[d] += ResultsOfIntegration[i, d];
            }
        }
    ).Execute();

    for(int d = 0; d < 3; d++) {
        centerOfMass[ts,d] = dropCoM[d] / dropVolume;
    }

    // minimal drop position
    double minZCoordDrop = 1.0;
    MultidimensionalArray InterfacePoints = XNSEUtils.GetInterfacePoints(LsTrk, levelSet);
    for(int i = 0; i < InterfacePoints.Lengths[0]; i++) {
        double zCoord = InterfacePoints[i, 2];
        if (zCoord < minZCoordDrop)
            minZCoordDrop = zCoord;
    }
    minDropPos[ts] = minZCoordDrop;

    // // sphericity
    // double dropSurface = 0.0;
    // CellQuadratureScheme cqs = SchemeHelper.GetLevelSetquadScheme(0, LsTrk.Regions.GetCutCellMask());
    // CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
    //     cqs.Compile(LsTrk.GridDat, quadOrder),
    //     delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
    //         EvalResult.SetAll(1.0);
    //     },
    //     delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
    //         for (int i = 0; i < Length; i++)
    //         dropSurface += ResultsOfIntegration[i, 0];
    //     }
    // ).Execute();

    // sphericity[ts] = Math.Pow(Math.PI, 1.0/3.0) * Math.Pow(6*dropVolume, 2.0/3.0) / dropSurface;

}

In [ ]:
centerOfMass

Dimension,Lengths,Storage,IsContinuous,StructureType,Length,NoOfCols,NoOfRows,IsLocked
2,"[ 29, 3 ]","[ 0.0779999999998405, 2.558302989516735E-14, 0.0006000009202340767, 0.07800000913097228, 1.141143452576779E-08, 0.00058877673692869, 0.07800004582745412, 6.165827891578793E-08, 0.0005775597410619704, 0.0780001068095097, 1.4472221693822214E-07, 0.0005663245828036336, 0.07800017816814982, 2.4068444588071557E-07, 0.0005550123300021596, 0.07800025632900907, 3.471196188637224E-07, 0.0005435805512960328, 0.07800037765669761, 5.178751200431484E-07, 0.0005318338165095191, 0.07800053130659605, 7.484683299922748E-07, 0.0005195933496623941, 0.07800058423527083, 8.374319550857631E-07, 0.0005078902749297473, 0.07800069705703166, 1.09371803060473E-06, 0.0004959659491326437, 0.07800079562791382, 1.4111064471417318E-06, 0.00048422340447693067, 0.07800090965978712, 1.7843403922798256E-06, 0.00047230633802986693, 0.07800104657871107, 2.1802657803354298E-06, 0.0004603550717183633, 0.07800119023167944, 2.602109620116846E-06, 0.00044839098124407575, 0.07800134377474834, 3.0654891316834105E-06, 0.0004363608212228924, 0.07800158224409139, 3.4252175995495027E-06, 0.00042436068997999657, 0.07800204950532191, 3.7130648090299577E-06, 0.0004125044613356929, 0.07800273355766432, 4.039330465528441E-06, 0.0004008267485342021, 0.07800366983860615, 4.537554218702028E-06, 0.0003894459529264859, 0.07800525085960763, 5.240186913454737E-06, 0.00037870948902141453, 0.07800573312982935, 5.61199870197617E-06, 0.0003677452049509152, 0.07800720861689761, 7.158742380570249E-06, 0.000358033031406997, 0.07800813141674631, 8.929575783688386E-06, 0.000344502787288337, 0.07800966818416495, 1.1550810168460266E-05, 0.0003298664995958589, 0.07801139299269286, 1.5061336944306873E-05, 0.0003156716255310446, 0.07801217460644314, 1.790644864682405E-05, 0.00030817251857992456, 0.0780128005392319, 2.023097874636097E-05, 0.00030049766689954085, 0.07801342888964685, 2.1920199429047637E-05, 0.0002924148157341421, 0.07801406712707451, 2.3229394408869595E-05, 0.0002842664003510255 ]",True,General,87,3,29,False


In [ ]:
var fmt = new PlotFormat();    
fmt.LineColor = LineColors.Blue;

var gp = new Gnuplot();
gp.SetMultiplot(1,2);

Plot2Ddata plt = new Plot2Ddata(); 
plt.AddDataGroup("centerOfMassZ", time, centerOfMass.ExtractSubArrayShallow(-1,2).To1DArray(), fmt);
gp.SetSubPlot(0, 0, "centerOfMassZ");
plt.ToGnuplot(gp);

// plt = new Plot2Ddata(); 
// plt.AddDataGroup("minimal Z position", time, minDropPos, fmt);
// gp.SetSubPlot(0, 1, "minZposDrop");
// plt.ToGnuplot(gp);

// plt = new Plot2Ddata(); 
// plt.AddDataGroup("sphericity", time, sphericity, fmt);
// gp.SetSubPlot(0, 1, "sphericity");
// plt.ToGnuplot(gp);

//gp.PlotNow()
gp.PlotSVG(xRes:1500,yRes:500)

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.0002 
 
 
 
 
 0.00025 
 
 
 
 
 0.0003 
 
 
 
 
 0.00035 
 
 
 
 
 0.0004 
 
 
 
 
 0.00045 
 
 
 
 
 0.0005 
 
 
 
 
 0.00055 
 
 
 
 
 0.0006 
 
 
 
 
 0.00065 
 
 
 
 
 0 
 
 
 
 
 5x10 -5 
 
 
 
 
 0.0001 
 
 
 
 
 0.00015 
 
 
 
 
 0.0002 
 
 
 
 
 0.00025 
 
 
 
 
 0.0003 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 centerOfMassZ 
 
 
 centerOfMassZ 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M475.5,62.1 L528.9,62.1 M94.5,87.5 L114.4,98.0 L134.4,108.4 L154.3,118.8 L174.2,129.3 L194.2,140.0
 L214.1,150.9 L234.1,162.2 L254.0,173.1 L273.9,184.2 L293.9,195.1 L313.8,206.2 L333.7,217.3 L353.7,228.4
 L373.6,239.5 L393.5,250.7 L413.5,261.7 L433.4,272.6 L453.4,283.1 L473.3,293.1 L493.2,303.3 L513.2,312.3
 L533.1,324.9 L553.0,338.5 L573.0,351.7 L583.9,358.6 L593.9,365.7 L603.9,373.3 L613.9,380.8 L622.8,388.6
 L632.8,396.7 L642.8,404.6 L652.7,409.8 L662.7,412.8 L672.7,414.0 '/>

In [ ]:
sess.Name

DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart4

In [ ]:
plt.SaveToTextFile("results/" + sess.Name + "_CenterOfMassZ")

## mean vertical velocity / lift over time

In [ ]:
double[] time = new double[timesteps];
double[] meanVertVelocity = new double[timesteps];
// double[] lift = new double[timesteps];
MultidimensionalArray pressureForce = MultidimensionalArray.Create(timesteps, 3);

In [ ]:
int degPhi = sess.Timesteps.ElementAt(0).GetField("Phi").Basis.Degree;
int quadOrder = (2 * degPhi) * 2 + 1;   // using Saye

In [ ]:
for (int ts = 0; ts < timesteps; ts++) {
    var tStep = sessTimesteps.ElementAt(ts);

    time[ts] = tStep.PhysicalTime;

    SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
    GridData grdData = (GridData)phi.GridDat; 
    LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
    levelSet.Acc(1.0, phi); 
    LevelSetTracker LsTrk = new LevelSetTracker(grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, levelSet);
    LsTrk.UpdateTracker(tStep.PhysicalTime);

    var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), quadOrder, 1).XQuadSchemeHelper;
    SpeciesId spcId = LsTrk.GetSpeciesId("A");
    var vqs = SchemeHelper.GetVolumeQuadScheme(spcId);


    // volume
    double dropVolume = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        vqs.Compile(LsTrk.GridDat, quadOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            EvalResult.SetAll(1.0);
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < Length; i++)
                dropVolume += ResultsOfIntegration[i, 0];
        }
    ).Execute();


    // mean vertical velocity
    double vertVel = 0.0;
    DGField velocityZ = ((XDGField)tStep.GetField("VelocityZ")).GetSpeciesShadowField("A");
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        vqs.Compile(LsTrk.GridDat, quadOrder),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            velocityZ.Evaluate(i0, Length, QR.Nodes, EvalResult.ExtractSubArrayShallow(-1,-1,0), 1.0);
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < Length; i++)
                vertVel += ResultsOfIntegration[i, 0];
        }
    ).Execute();

    meanVertVelocity[ts] = vertVel / dropVolume;


    // // lift
    // double[] pForce = new double[3];
    // DGField pressure = ((XDGField)tStep.GetField("Pressure")).GetSpeciesShadowField("B");
    // CellQuadratureScheme cqs = SchemeHelper.GetLevelSetquadScheme(0, LsTrk.Regions.GetCutCellMask());
    // CellQuadrature.GetQuadrature(new int[] { 3 }, LsTrk.GridDat,
    //     cqs.Compile(LsTrk.GridDat, quadOrder),
    //     delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
    //         var normals = LsTrk.DataHistories[0].Current.GetLevelSetNormals(QR.Nodes, i0, Length);
    //         int K = EvalResult.GetLength(1);
    //         var pressureEval = MultidimensionalArray.Create(Length, K);
    //         pressure.Evaluate(i0, Length, QR.Nodes, pressureEval, 1.0);  
    //         for(int d = 0; d < 3; d++) {              
    //             for (int j = 0; j < Length; j++) {
    //                 for (int k = 0; k < K; k++) {
    //                     EvalResult[j, k, d] = pressureEval[j, k] * normals[j, k, d];
    //                 }
    //             }
    //         }
                
    //     },
    //     delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
    //         for(int d = 0; d < 3; d++) {  
    //             for (int i = 0; i < Length; i++)
    //                 pForce[d] += ResultsOfIntegration[i, d];
    //         }
    //     }
    // ).Execute();

    // for(int d = 0; d < 3; d++)
    //     pressureForce[ts, d] = pForce[d];

}

In [ ]:
var fmt = new PlotFormat();    
fmt.LineColor = LineColors.Blue;

var gp = new Gnuplot();
gp.SetMultiplot(1,2);

Plot2Ddata plt = new Plot2Ddata(); 
plt.AddDataGroup("meanVelocityZ", time, meanVertVelocity, fmt);
gp.SetSubPlot(0, 0, "meanVelocityZ");
plt.ToGnuplot(gp);

// plt = new Plot2Ddata(); 
// plt.AddDataGroup("lift", time, pressureForce.ExtractSubArrayShallow(-1,2).To1DArray(), fmt);
// gp.SetSubPlot(0, 1, "lift");
// plt.ToGnuplot(gp);

//gp.PlotNow()
gp.PlotSVG(xRes:1500,yRes:500)

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -1.26 
 
 
 
 
 -1.24 
 
 
 
 
 -1.22 
 
 
 
 
 -1.2 
 
 
 
 
 -1.18 
 
 
 
 
 -1.16 
 
 
 
 
 -1.14 
 
 
 
 
 -1.12 
 
 
 
 
 0 
 
 
 
 
 5x10 -5 
 
 
 
 
 0.0001 
 
 
 
 
 0.00015 
 
 
 
 
 0.0002 
 
 
 
 
 0.00025 
 
 
 
 
 0.0003 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 meanVelocityZ 
 
 
 meanVelocityZ

In [ ]:
plt.SaveToTextFile("results/" + sess.Name + "_meanVelocityZ")